# 08_04 Campaign-level action features: depth, flight position, per-article lift

## Mechanism and expected effect (stated before running)

The model sees promotions only as a binary flag (`action_on_forecast_day`) plus
one pooled lift number per sourcing group. The raw sale lines carry what the
flag throws away: the campaign identity (`AKTIONSNUMMER`), its **planned
validity window** (`GUELTIG_VON`/`GUELTIG_BIS`) and the **promoted unit
price** — extracted once into `data/interim/actions/` by
`src/data/preparation/extract_action_history.py` (2.44 M action
article-store-days; 285 k article-day regular prices). Three mechanisms should
make these informative:

1. **Discount depth.** A 38 %-off campaign and a token action are currently
   identical to the model; measured median depth is 21.7 % with p10 ≈ 0 and
   p90 ≈ 38 %, so the response the model averages over is highly heterogeneous.
2. **Flight position.** Campaigns run as Mon–Sat flights; pantry-loading and
   ad-exposure make day 1 differ from day 6.
3. **Per-article response.** Promotion lift differs by article far more than
   the single pooled `mean_action_lift_in_sourcing_group` can express.

**Pre-registered evidence** (baseline `run_fixed_data` forecasts on
`transactions_fixed`, own-action rows of the 20-origin evaluation window):

- **Depth is the dominant signal**: bias is **+11.6 %** on token actions
  (< 10 % off), +7.2 % at 10–25 %, and **−5.2 %** on deep actions (≥ 25 %) —
  a monotone 17 pp swing, with the deep bucket carrying 1.16 M kg of the
  action volume. Exactly the signature of one average response applied to
  heterogeneous discounts.
- **Flight day** shows a mild monotone drift (−1.9 % day 1 → −3.8 % day 6)
  plus a pathological past-planned-window bucket (+86 % bias, 3.2 k kg).
- **Article lift terciles** drift −5.3 % → +0.8 % from the least to the most
  promotion-responsive articles.

**Budget**: the depth gradient alone spans ~3 % of total volume in signed
bias, so a realistic expectation is **−0.15 to −0.5 pp row WAPE**, with pooled
bias moving toward zero (the underforecast deep bucket dominates the volume).
Pass/fail: a row gain clear of the 0.053 pp seed floor **without** pooled bias
degrading beyond the 0.19 pp bias floor, and the depth-bucket bias gradient
visibly flattening.

In [1]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import TWO_STAGE_MODEL_NAME
from src.models.results import result_path

pd.set_option('display.max_columns', 40)

design = load_benchmark_design()
BASELINE_FORECASTS = (ROOT / 'reports' / 'results' / 'run_fixed_data' / 'forecasts'
                      / 'artikel_markt_multi7days_lightgbm_two_stage.csv')
NEW_FORECASTS = result_path(TWO_STAGE_MODEL_NAME, design)

con = duckdb.connect()
con.execute('PRAGMA threads=4')
con.execute('''
    CREATE OR REPLACE TEMP TABLE base AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
           actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
           action_on_forecast_day::INTEGER AS own_action
    FROM read_csv_auto(?) WHERE is_active
''', [str(BASELINE_FORECASTS)])
con.execute('''
    CREATE OR REPLACE TEMP TABLE act AS
    SELECT ARTIKEL_ID, period::DATE AS period, MIN(gueltig_von)::DATE AS gueltig_von,
           AVG(action_unit_price) AS price
    FROM read_parquet(?) GROUP BY 1, 2
''', [str(ROOT / 'data' / 'interim' / 'actions' / 'article_store_day_actions.parquet')])
con.execute('''
    CREATE OR REPLACE TEMP TABLE depth AS
    SELECT act.ARTIKEL_ID, act.period,
           1.0 - act.price / NULLIF(AVG(reg.regular_unit_price), 0) AS d
    FROM act
    JOIN read_parquet(?) AS reg
        ON reg.ARTIKEL_ID = act.ARTIKEL_ID
        AND reg.period::DATE < act.period
        AND reg.period::DATE >= act.period - INTERVAL 28 DAY
    GROUP BY act.ARTIKEL_ID, act.period, act.price
''', [str(ROOT / 'data' / 'interim' / 'actions' / 'article_day_regular_price.parquet')])
print(con.execute('SELECT COUNT(*), COUNT(DISTINCT origin) FROM base').fetchone())

(2498967, 20)


## Mechanism verification on the baseline forecasts

Own-action rows of the fixed 20-origin window, bucketed by the campaign
attributes the model cannot currently see.

In [2]:
DEPTH_BUCKET = ("CASE WHEN d < 0.1 THEN '1 token (<10%)' "
                "WHEN d < 0.25 THEN '2 mid (10-25%)' ELSE '3 deep (>=25%)' END")
by_depth = con.execute(f'''
    SELECT {DEPTH_BUCKET} AS depth_bucket, COUNT(*) AS rows_n, SUM(actual) AS volume,
           SUM(forecast - actual) / SUM(actual) AS rel_bias,
           SUM(ABS(forecast - actual)) / SUM(actual) AS wape
    FROM base JOIN depth USING (ARTIKEL_ID, period)
    WHERE own_action = 1 GROUP BY 1 ORDER BY 1
''').fetchdf()
display(by_depth.style.format({'rows_n': '{:,.0f}', 'volume': '{:,.0f}',
                               'rel_bias': '{:+.2%}', 'wape': '{:.2%}'}))
by_flight = con.execute('''
    SELECT LEAST(DATE_DIFF('day', act.gueltig_von, base.period) + 1, 7) AS flight_day,
           COUNT(*) AS rows_n, SUM(actual) AS volume,
           SUM(forecast - actual) / SUM(actual) AS rel_bias,
           SUM(ABS(forecast - actual)) / SUM(actual) AS wape
    FROM base JOIN act USING (ARTIKEL_ID, period)
    WHERE own_action = 1 AND base.period >= act.gueltig_von
    GROUP BY 1 ORDER BY 1
''').fetchdf()
display(by_flight.style.format({'rows_n': '{:,.0f}', 'volume': '{:,.0f}',
                                'rel_bias': '{:+.2%}', 'wape': '{:.2%}'}))

,depth_bucket,rows_n,volume,rel_bias,wape
0,1 token (<10%),"32,955","43,339",+11.57%,57.41%
1,2 mid (10-25%),"90,054","247,999",+7.21%,56.82%
2,3 deep (>=25%),"166,546","1,161,857",-5.17%,40.77%


,flight_day,rows_n,volume,rel_bias,wape
0,1,"42,971","188,397",-1.92%,46.02%
1,2,"47,856","199,429",-0.83%,45.62%
2,3,"49,453","219,012",-0.29%,44.43%
3,4,"47,523","249,130",-5.37%,43.73%
4,5,"46,571","270,673",-3.07%,42.61%
5,6,"51,444","323,377",-3.80%,42.26%
6,7,"3,737","3,177",+86.17%,112.68%


## The features

Six columns join the contract (`FEATURE_BUILDER_VERSION` 2026-08-07.4 →
2026-08-16.3, all partitions rebuilt on `transactions_fixed`, only the
two-stage model refit; both stages receive all six):

- **`action_depth_on_forecast_day`** — 1 − promoted price ÷ the article's mean
  regular price of the 28 days strictly before the origin; the promoted price
  and validity window belong to the same known-ahead schedule as
  `action_on_forecast_day`. Zero without an action.
- **`action_flight_day`** — target date − `GUELTIG_VON` + 1 (capped at 28);
  zero without an action.
- **`article_action_lift`** — per-article action/regular demand ratio − 1 over
  strictly pre-origin active rows, EB-shrunk toward
  `mean_action_lift_in_sourcing_group` with a 24-observation prior.
- **`action_expected_lift`** — the `06_02`-style precomputed interaction:
  `article_action_lift` on promoted rows, zero otherwise.
- **`article_typical_action_depth`** — mean pre-origin discount depth of the
  article's own campaigns.
- **`same_class_promoted_depth`** — deepest typical depth among same-Warenklasse
  articles promoted at the store that day (the intensity version of the
  reverted `08_03` breadth features).

Unit tests cover the pre-origin-only depth reference (a post-origin price
canary), the flight-day arithmetic, the exact shrinkage value, the pooled
fallback for articles without action history, and class/quiet-day degradation
(91 tests pass).

## Results

Both runs cover the same 20 origins on `transactions_fixed`; the baseline is
the published `run_fixed_data` two-stage fit (seed 42, identical configuration
apart from the six new features).

In [3]:
con.execute('''
    CREATE OR REPLACE TEMP TABLE new_run AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
           actual::DOUBLE AS actual, forecast::DOUBLE AS forecast
    FROM read_csv_auto(?) WHERE is_active
''', [str(NEW_FORECASTS)])

GRAINS = {
    'row': None,
    'article-store-week': 'ARTIKEL_ID, MARKT_ID, origin',
    'article-day': 'ARTIKEL_ID, origin, period',
}


def score(table):
    out = {'run': table}
    for grain, keys in GRAINS.items():
        inner = (f'SELECT actual, forecast FROM {table}' if keys is None else
                 f'SELECT SUM(actual) AS actual, SUM(forecast) AS forecast '
                 f'FROM {table} GROUP BY {keys}')
        wape, bias = con.execute(
            f'SELECT SUM(ABS(forecast - actual)) / SUM(actual), '
            f'SUM(forecast - actual) / SUM(actual) FROM ({inner})').fetchone()
        out[grain] = wape
        if grain == 'row':
            out['bias'] = bias
    return out


comparison = pd.DataFrame([score('base'), score('new_run')])
comparison['run'] = ['baseline (run_fixed_data)', 'with campaign features']
for g in GRAINS:
    comparison[f'Δ {g} (pp)'] = (comparison[g] - comparison[g].iloc[0]) * 100
display(comparison.style.format({**{g: '{:.2%}' for g in GRAINS}, 'bias': '{:+.2%}',
                                 **{f'Δ {g} (pp)': '{:+.2f}' for g in GRAINS}}))

per_origin = con.execute('''
    SELECT new_run.origin,
           SUM(ABS(base.forecast - base.actual)) / SUM(base.actual) AS wape_base,
           SUM(ABS(new_run.forecast - new_run.actual)) / SUM(new_run.actual) AS wape_new
    FROM new_run JOIN base USING (ARTIKEL_ID, MARKT_ID, origin, period)
    GROUP BY 1 ORDER BY 1
''').fetchdf()
per_origin['delta_pp'] = (per_origin.wape_new - per_origin.wape_base) * 100
print(f'origins improved: {(per_origin.delta_pp < 0).sum()} / {len(per_origin)}')
display(per_origin.style.format({'wape_base': '{:.2%}', 'wape_new': '{:.2%}',
                                 'delta_pp': '{:+.2f}'}))

,run,row,bias,article-store-week,article-day,Δ row (pp),Δ article-store-week (pp),Δ article-day (pp)
0,baseline (run_fixed_data),57.44%,-1.21%,37.12%,23.96%,+0.00,+0.00,+0.00
1,with campaign features,56.92%,-0.25%,36.40%,23.41%,-0.53,-0.73,-0.54


origins improved: 14 / 20


,origin,wape_base,wape_new,delta_pp
0,2026-03-02 00:00:00,55.63%,54.92%,-0.71
1,2026-03-09 00:00:00,60.93%,59.60%,-1.33
2,2026-03-16 00:00:00,57.49%,55.93%,-1.56
3,2026-03-23 00:00:00,62.05%,59.93%,-2.12
4,2026-03-30 00:00:00,48.71%,48.49%,-0.22
5,2026-04-06 00:00:00,58.53%,58.57%,+0.04
6,2026-04-13 00:00:00,67.63%,67.90%,+0.27
7,2026-04-20 00:00:00,58.43%,58.89%,+0.46
8,2026-04-27 00:00:00,54.61%,57.05%,+2.44
9,2026-05-04 00:00:00,56.68%,55.03%,-1.64


In [4]:
# Did the depth gradient flatten? Same buckets, new forecasts.
closed = con.execute(f'''
    SELECT {DEPTH_BUCKET} AS depth_bucket,
           SUM(base.forecast - base.actual) / SUM(base.actual) AS bias_base,
           SUM(new_run.forecast - new_run.actual) / SUM(new_run.actual) AS bias_new,
           SUM(ABS(base.forecast - base.actual)) / SUM(base.actual) AS wape_base,
           SUM(ABS(new_run.forecast - new_run.actual)) / SUM(new_run.actual) AS wape_new
    FROM base
    JOIN depth USING (ARTIKEL_ID, period)
    JOIN new_run USING (ARTIKEL_ID, MARKT_ID, origin, period)
    WHERE base.own_action = 1
    GROUP BY 1 ORDER BY 1
''').fetchdf()
display(closed.style.format({'bias_base': '{:+.2%}', 'bias_new': '{:+.2%}',
                             'wape_base': '{:.2%}', 'wape_new': '{:.2%}'}))

importance = pd.read_csv(
    ROOT / 'reports' / 'results' / 'feature_importance'
    / 'artikel_markt_multi7days_lightgbm_two_stage.csv')
new_features = ['action_depth_on_forecast_day', 'action_flight_day',
                'article_action_lift', 'action_expected_lift',
                'article_typical_action_depth', 'same_class_promoted_depth']
importance['rank'] = importance.groupby(['evaluation_origin', 'stage'])['gain'] \
    .rank(ascending=False)
summary = (importance[importance.feature.isin(new_features)]
           .groupby(['stage', 'feature'])
           .agg(mean_rank=('rank', 'mean'), mean_gain_share=('gain_share', 'mean'))
           .reset_index())
print('features per stage:',
      importance.groupby(['evaluation_origin', 'stage'])['feature'].nunique().max())
display(summary.style.format({'mean_rank': '{:.1f}', 'mean_gain_share': '{:.3%}'}))

,depth_bucket,bias_base,bias_new,wape_base,wape_new
0,1 token (<10%),+11.57%,+3.41%,57.41%,54.22%
1,2 mid (10-25%),+7.21%,-1.35%,56.82%,54.00%
2,3 deep (>=25%),-5.17%,-0.89%,40.77%,40.52%


features per stage: 58


,stage,feature,mean_rank,mean_gain_share
0,occurrence,action_depth_on_forecast_day,12.6,1.550%
1,occurrence,action_expected_lift,6.8,3.249%
2,occurrence,action_flight_day,3.8,5.005%
3,occurrence,article_action_lift,30.0,0.070%
4,occurrence,article_typical_action_depth,38.0,0.040%
5,occurrence,same_class_promoted_depth,28.6,0.073%
6,positive_quantity,action_depth_on_forecast_day,9.4,2.112%
7,positive_quantity,action_expected_lift,2.8,12.655%
8,positive_quantity,action_flight_day,6.4,6.252%
9,positive_quantity,article_action_lift,23.0,0.306%


## Verdict

**Adopted. The largest feature-batch gain since `06_02`, and the first that
improves WAPE and bias together.**

Against the `07_02` seed floor (row sd 0.053 pp, bias sd 0.19 pp), on the same
dataset and configuration:

- **Row WAPE 57.44% → 56.92% (−0.53 pp, ≈ 10 sd)** — at the top of the
  pre-registered −0.15…−0.5 pp budget; article-store-week −0.73 pp and
  article-day −0.54 pp move with it, so nothing is being bought at another
  grain's expense. 14/20 origins improved.
- **Pooled bias −1.21% → −0.25% (≈ 5 sd, toward zero)** — the best bias state
  of the entire programme, and the direction the volume-dominant deep-discount
  bucket predicted.
- **The pre-registered signature appeared exactly**: the depth-bucket bias
  gradient flattened from +11.6% / +7.2% / −5.2% to **+3.4% / −1.4% / −0.9%**,
  with token- and mid-depth WAPE dropping ~3 pp each.
- **The channel lesson is confirmed a fourth time**: the precomputed
  interaction `action_expected_lift` is the workhorse (quantity-stage rank
  **2.8/58**, 12.7% gain share; occurrence 6.8/58), flanked by
  `action_flight_day` (occurrence rank 3.8, 5.0%) and
  `action_depth_on_forecast_day` (9.4/12.6). The raw ingredients
  (`article_action_lift`, `article_typical_action_depth`) rank low precisely
  because the interaction absorbed their role, and `same_class_promoted_depth`
  is marginal (≤ 0.07%) — consistent with `08_03`'s finding that the
  competition channel carries little; the value was always in the article's
  *own* campaign attributes.

Caveats: single seed (though at 10 sd the row effect is far clear of the
floor); one adverse origin (2026-04-27, +2.44 pp) worth a look in a future
error review; `same_class_promoted_depth` is a candidate for removal in a
later cleanup batch if it stays inert.